# 7. Day Simulation Discussion

By the time the workflow reaches the day runner, the narrative has already split into two reusable artifacts: a scheduler-facing user table over the day, and a per-user lookup process that turns each active bin into a TDMA problem. This notebook shows that orchestration step explicitly.


## 1. Rebuild the day-wide scheduler-facing demand table

The day simulation first expands the hourly load curve into quarter-hour bins, generates synthetic sessions, and then projects those sessions onto the lean scheduler contract used by the run layer.


In [ ]:
import sys
from pathlib import Path
from IPython.display import display

repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent

for path in (repo_root / "notebooks", repo_root / "src"):
    resolved = str(path.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)

from configs.day_cycle import (
    DEFAULT_DAY_CYCLE_LOAD_CURVE_CSV,
    DEFAULT_SYNTHETIC_SESSION_GENERATION_CONFIG,
)
from helpers.day_simulation_helpers import build_day_simulation_artifacts
from models import PASwitchPolicy

load_curve_csv = Path(DEFAULT_DAY_CYCLE_LOAD_CURVE_CSV)
if not load_curve_csv.is_absolute():
    load_curve_csv = repo_root / load_curve_csv

artifacts = build_day_simulation_artifacts(
    load_curve_csv=load_curve_csv,
    session_generation_config=DEFAULT_SYNTHETIC_SESSION_GENERATION_CONFIG,
    switch_policy=PASwitchPolicy.STANDBY,
    target_user_count=4,
)


In [ ]:
display(artifacts.bin_summary_table.head(12))


## 2. Solve one representative bin through the day-run interface

`run_bin` is the smallest complete day-run unit. It receives the already-normalized user table for one bin, builds the trusted per-user spaces, runs the joint scheduler, and returns only the lean schedule summary that the day export keeps.


In [ ]:
print(f"Example solved bin: {artifacts.example_bin_index}")
display(artifacts.example_user_table)
display(artifacts.example_schedule_summary)
display(artifacts.example_allocation_table)


The final notebook no longer needs to explain how the day is simulated. It can focus on the exported full-day results, compare switch policies, and discuss the system-level energy consequences of the resource-allocation choices.
